# 00 - Setup

This project trains a player-churn classifier on SageMaker. Three tools, three
jobs, no overlap:

| Concern | Tool | Source of truth |
| --- | --- | --- |
| Code | git / GitHub | the commit |
| Data | **DVC** (S3 remote) | `dvc.lock` + `*.dvc` pointers |
| Experiments & models | **SageMaker managed MLflow** | the MLflow run / registered model |

The join between them: every MLflow run is tagged with the git commit **and** the
DVC content hash of the data it consumed. Given any run you can reconstruct its
exact inputs with `git checkout <commit> && dvc checkout`.

What this notebook does:
1. Installs the project as an editable package.
2. Confirms your AWS identity, region and bucket.
3. Finds or creates the managed MLflow tracking server.
4. Writes a `.env` file so **no later notebook asks you to paste an ARN**.

Run it once per environment.

## 1. Install

`pip install -e ..` installs the `churn` package from `src/`, so `python -m churn.features`
works from the project root and from CI identically.

In [1]:
%pip install -q -e .. -r ../requirements-dev.txt

/Users/ritabanchowdhury/Desktop/mlops-sagemaker-mlflow/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
# Restart the kernel after the install above, then continue from here.
import json
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)          # every command in these notebooks runs from the project root
os.environ["CHURN_PROJECT_ROOT"] = str(PROJECT_ROOT)

# Put this kernel's venv bin/ first on PATH so shell commands (dvc, ...) resolve to the
# venv, however Jupyter was launched.
import sys
os.environ["PATH"] = f"{Path(sys.executable).parent}{os.pathsep}{os.environ['PATH']}"
print("project root:", PROJECT_ROOT)

project root: /Users/ritabanchowdhury/Desktop/mlops-sagemaker-mlflow/churn-mlops


In [2]:
import boto3
import mlflow
import sagemaker

from src import churn

print("churn     ", churn.__version__)
#print("sagemaker ", sagemaker.__version__)
print("boto3     ", boto3.__version__)
print("mlflow    ", mlflow.__version__)
print("dvc       ", subprocess.run(["dvc", "version"], capture_output=True, text=True).stdout.splitlines()[0])

/Users/ritabanchowdhury/Desktop/mlops-sagemaker-mlflow/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sagemaker.config INFO - Not applying SDK defaults from location: /usr/local/share/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/ritabanchowdhury/Library/Application Support/sagemaker/config.yaml


/Users/ritabanchowdhury/Desktop/mlops-sagemaker-mlflow/.venv/lib/python3.11/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


churn      0.1.0
boto3      1.43.117
mlflow     3.17.2
dvc        DVC version: 3.67.1 (pip)


## 2. AWS identity, region, bucket

`Settings.load()` reads env vars first and falls back to the SageMaker defaults,
so this works unchanged in Studio and on a GitHub Actions runner.

In [3]:
from src.churn.config import load_params

params = load_params()
REGION = params["aws"]["region"]
os.environ["AWS_DEFAULT_REGION"] = REGION

identity = boto3.client("sts", region_name=REGION).get_caller_identity()
print("account:", identity["Account"])
print("caller :", identity["Arn"])
print("region :", REGION)

account: 063676201336
caller : arn:aws:iam::063676201336:user/test_mlops
region : us-east-1


In [ ]:
import os

# Outside SageMaker the caller is an IAM user, so get_execution_role() cannot work and the role
# ARN has to be supplied. Never hard-code it here: this notebook is committed.
# Lookup order: CHURN_SAGEMAKER_ROLE env var -> .env from a previous run -> SageMaker -> prompt.
env_file = PROJECT_ROOT / ".env"
if not os.getenv("CHURN_SAGEMAKER_ROLE") and env_file.exists():
    for line in env_file.read_text().splitlines():
        key, _, value = line.partition("=")
        if key == "CHURN_SAGEMAKER_ROLE" and value:
            os.environ[key] = value

In [ ]:
sm_session = sagemaker.Session(boto_session=boto3.Session(region_name=REGION))
BUCKET = sm_session.default_bucket()
try:
    ROLE = os.getenv("CHURN_SAGEMAKER_ROLE") or sagemaker.get_execution_role(sagemaker_session=sm_session)
except ValueError:
    ROLE = input("SageMaker execution role ARN: ").strip()

print("bucket:", BUCKET)
print("role  :", ROLE)

## 3. MLflow tracking server

`ensure_tracking_server` reuses a server in `Created` *or* `Creating` state before
making a new one, so re-running this notebook does not pile up servers (each one
bills hourly).

`AutomaticModelRegistration=True` is deliberate: it is what mirrors
MLflow-registered model versions into the SageMaker Model Registry, which is how
notebook 06's deploy path keeps working while you only ever talk to MLflow.

Creation takes **~25 minutes**. You can carry on with notebooks 01 and 02 (which
need no MLflow) while it provisions.

In [8]:
from src.churn.aws import ensure_tracking_server

MLFLOW_ARN, MLFLOW_NAME = ensure_tracking_server(REGION, BUCKET, ROLE)
print("MLflow server:", MLFLOW_NAME)
print("ARN          :", MLFLOW_ARN)

MLflow server: mlflow-churn-28-15-00-16
ARN          : arn:aws:sagemaker:us-east-1:063676201336:mlflow-tracking-server/mlflow-churn-28-15-00-16


In [10]:
# Blocks until the server reports `Created`. Skip if you want to go run 01 and 02 first.
from src.churn.aws import wait_for_tracking_server

print(wait_for_tracking_server(REGION, MLFLOW_NAME))

  MLflow server mlflow-churn-28-15-00-16: Creating
  MLflow server mlflow-churn-28-15-00-16: Creating
  MLflow server mlflow-churn-28-15-00-16: Creating
  MLflow server mlflow-churn-28-15-00-16: Creating
  MLflow server mlflow-churn-28-15-00-16: Creating
  MLflow server mlflow-churn-28-15-00-16: Creating
  MLflow server mlflow-churn-28-15-00-16: Creating
  MLflow server mlflow-churn-28-15-00-16: Creating
  MLflow server mlflow-churn-28-15-00-16: Creating
  MLflow server mlflow-churn-28-15-00-16: Creating
  MLflow server mlflow-churn-28-15-00-16: Creating
  MLflow server mlflow-churn-28-15-00-16: Creating
  MLflow server mlflow-churn-28-15-00-16: Creating
  MLflow server mlflow-churn-28-15-00-16: Creating
  MLflow server mlflow-churn-28-15-00-16: Creating
  MLflow server mlflow-churn-28-15-00-16: Creating
  MLflow server mlflow-churn-28-15-00-16: Creating
  MLflow server mlflow-churn-28-15-00-16: Creating
  MLflow server mlflow-churn-28-15-00-16: Creating
  MLflow server mlflow-churn-28

## 4. Write `.env`

This is the only place these values are stored. Later notebooks call
`load_dotenv()`; CI sets the same names as GitHub Actions variables and secrets.

`.env` is gitignored -- an account id is not repo content.

In [11]:
env_path = PROJECT_ROOT / ".env"
env_path.write_text(
    "\n".join(
        [
            f"AWS_DEFAULT_REGION={REGION}",
            f"CHURN_S3_BUCKET={BUCKET}",
            f"CHURN_SAGEMAKER_ROLE={ROLE}",
            f"MLFLOW_TRACKING_URI={MLFLOW_ARN}",
            "",
        ]
    ),
    encoding="utf-8",
)
print(env_path.read_text())

AWS_DEFAULT_REGION=us-east-1
CHURN_S3_BUCKET=sagemaker-us-east-1-063676201336
CHURN_SAGEMAKER_ROLE=arn:aws:iam::063676201336:role/service-role/AmazonSageMaker-ExecutionRole-20261028T105273
MLFLOW_TRACKING_URI=arn:aws:sagemaker:us-east-1:063676201336:mlflow-tracking-server/mlflow-churn-28-15-00-16



In [13]:
# Confirm Settings picks everything up.
from src.churn.config import Settings

for key, value in [
    line.split("=", 1) for line in env_path.read_text().strip().splitlines()
]:
    os.environ[key] = value

settings = Settings.load(params)
print("s3 base   :", settings.s3_base)
print("dvc remote:", settings.dvc_remote_url)
print("mlflow uri:", settings.mlflow_tracking_uri)

s3 base   : s3://sagemaker-us-east-1-063676201336/player-churn/dvc-mlflow
dvc remote: s3://sagemaker-us-east-1-063676201336/player-churn/dvc-store
mlflow uri: arn:aws:sagemaker:us-east-1:063676201336:mlflow-tracking-server/mlflow-churn-28-15-00-16


## 5. Open the MLflow UI

Studio: **Applications → MLflow**, or use the presigned URL below.

In [14]:
sm = boto3.client("sagemaker", region_name=REGION)
print(sm.create_presigned_mlflow_tracking_server_url(TrackingServerName=MLFLOW_NAME)["AuthorizedUrl"])

https://t-cqpjjt4ftawu.us-east-1.experiments.sagemaker.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IjRkZWEzZmJhLWI5N2MtNDI5NC1iMDMzLWQwNWY1OTZiNzUxNiIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNDRiemJ4NVUxa3JWbzdzU1RLbFE0TDl1RFpWaXVOT3RlNUVQazFKUmF2TzBBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGeVREWjZhV1J1YVhVNFFXcExjSFZSVGpsc1pHODBhbGR6YVZKSVQwOXdWelUyY1hNeE1VRmtZemRqVEV4c1ZsSnBkM0p6YlhCTWIyczBXSFJQVEhWd1VUMDlBQUVBQjJGM2N5MXJiWE1BUzJGeWJqcGhkM002YTIxek9uVnpMV1ZoYzNRdE1UbzROVEUzTWpVME56TTFNelU2YTJWNUwyRTVZMkZrWVRjNUxXTTVaRFl0TkRGbU5TMDRNakl5TFRsak1qYzNaVGMyWkdRMllRQzRBUUlCQUhocndMdHlQbTE4RGVMeWFQbzJFRDZYTCtPRU5VVnpRUlJraDlnelJTMEtHUUZpN1crK0QrZzlOSllLdnpicVdTQXpBQUFBZmpCOEJna3Foa2lHOXcwQkJ3YWdiekJ0QWdFQU1HZ0dDU3FHU0liM0RRRUhBVEFlQmdsZ2hrZ0JaUU1FQVM0d0VRUU1JNnBQKzV6UnNwZ2xSM2NnQWdFUWdEdFI0MS9aMnBPZU1GNVhoczdJRWpUcXA2SnlQMGYwaE5lVzcveXRpNndhTHo3OWMvSTlNbFo3aWQ1WWJFTUpweE9nRkJjUUMvZVlwTFlvUGdJQUFCQUFib1lxdWd1Y1hLWms5aG1rQTM0MDFDUS9ybGVPaEJ2N0JYWjBPQW9yd2pUQXlEWG5XWStZSmxvOW1LY

---
## Next

**`01-data-versioning.ipynb`** -- put the dataset under DVC.

### What changed vs. the original workshop
`00-start-here.ipynb` ended by printing an ARN you then pasted into seven other
notebooks. Here it lands in `.env` once and `Settings.load()` finds it. The
`region = "us-east-1"` hardcoded in the old notebooks 02-05 now comes from
`params.yaml`.